In [ ]:
from datasets import load_dataset
dataset = load_dataset("WillHeld/top_v2", "default")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/559 [00:00<?, ?B/s]

data/train-00000-of-00001-4f5cf905029cbf(…):   0%|          | 0.00/6.73M [00:00<?, ?B/s]

data/test-00000-of-00001-deac2888ce8ad39(…):   0%|          | 0.00/2.02M [00:00<?, ?B/s]

data/eval-00000-of-00001-3ffa52405fac46a(…):   0%|          | 0.00/927k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/124597 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38785 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/17160 [00:00<?, ? examples/s]

In [ ]:
dataset['train'][0]

{'domain': 'alarm',
 'utterance': 'Set the alarm for tonight.',
 'semantic_parse': '[IN:CREATE_ALARM Set the alarm [SL:DATE_TIME for tonight ] . ]'}

In [ ]:
domains = set(dataset["train"]["domain"])
print(domains)

{'messaging', 'navigation', 'reminder', 'music', 'event', 'weather', 'alarm', 'timer'}


In [ ]:
import re

def extract_intents(example):
    return re.findall(r'IN:[A-Z_]+', example["semantic_parse"])

intents = set()

for example in dataset["train"]:
    intents.update(extract_intents(example))

print(intents)
print("Total intents:", len(intents))

{'IN:PAUSE_MUSIC', 'IN:GET_LOCATION_SCHOOL', 'IN:UPDATE_ALARM', 'IN:DELETE_TIMER', 'IN:PAUSE_TIMER', 'IN:GET_REMINDER_DATE_TIME', 'IN:CANCEL_MESSAGE', 'IN:GET_RECURRING_DATE_TIME', 'IN:DELETE_ALARM', 'IN:NEGATION', 'IN:GET_DISTANCE', 'IN:GET_INFO_ROAD_CONDITION', 'IN:UNSUPPORTED_MUSIC', 'IN:GET_EVENT', 'IN:UPDATE_REMINDER_TODO', 'IN:UNSUPPORTED_ALARM', 'IN:GET_LOCATION_WORK', 'IN:GET_TIMER', 'IN:GET_REMINDER_LOCATION', 'IN:IGNORE_MESSAGE', 'IN:UNSUPPORTED_MESSAGING', 'IN:GET_LOCATION', 'IN:GET_CONTACT', 'IN:LOOP_MUSIC', 'IN:GET_ESTIMATED_ARRIVAL', 'IN:GET_LOCATION_HOME', 'IN:UPDATE_REMINDER', 'IN:CREATE_TIMER', 'IN:GET_REMINDER_AMOUNT', 'IN:GET_ESTIMATED_DURATION', 'IN:GET_REMINDER', 'IN:GET_TODO', 'IN:UNSUPPORTED_TIMER', 'IN:GET_ESTIMATED_DEPARTURE', 'IN:ADD_TIME_TIMER', 'IN:ADD_TO_PLAYLIST_MUSIC', 'IN:RESTART_TIMER', 'IN:STOP_MUSIC', 'IN:GET_ALARM', 'IN:GET_INFO_ROUTE', 'IN:REPLAY_MUSIC', 'IN:UNSUPPORTED_NAVIGATION', 'IN:GET_SUNSET', 'IN:LIKE_MUSIC', 'IN:UNSUPPORTED_WEATHER', 'IN:SEN

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['domain', 'utterance', 'semantic_parse'],
        num_rows: 124597
    })
    test: Dataset({
        features: ['domain', 'utterance', 'semantic_parse'],
        num_rows: 38785
    })
    eval: Dataset({
        features: ['domain', 'utterance', 'semantic_parse'],
        num_rows: 17160
    })
})


In [ ]:
import pandas as pd

df = dataset["train"].to_pandas()

In [ ]:
df.to_csv("topv2_train.csv", index=False)

In [ ]:
relevant = ["CREATE_REMINDER", "GET_REMINDER",
            "DELETE_REMINDER", "CREATE_TIMER",
            "GET_EVENTS", "SNOOZE_REMINDER"]

In [ ]:
dataset_s = load_dataset("snips_built_in_intents")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/328 [00:00<?, ? examples/s]

In [ ]:
dataset_s['train'][0]

{'text': "Share my location with Hillary's sister", 'label': 5}

In [ ]:
if 'label' in dataset_s['train'].features:
    snips_label_names = dataset_s['train'].features['label'].names
    print("Labels and their meanings from 'snips_built_in_intents' dataset:")
    for i, label_name in enumerate(snips_label_names):
        print(f"- ID: {i}, Meaning: {label_name}")
else:
    print("Could not find 'label' feature in snips dataset to extract meanings.")

Labels and their meanings from 'snips_built_in_intents' dataset:
- ID: 0, Meaning: ComparePlaces
- ID: 1, Meaning: RequestRide
- ID: 2, Meaning: GetWeather
- ID: 3, Meaning: SearchPlace
- ID: 4, Meaning: GetPlaceDetails
- ID: 5, Meaning: ShareCurrentLocation
- ID: 6, Meaning: GetTrafficInformation
- ID: 7, Meaning: BookRestaurant
- ID: 8, Meaning: GetDirections
- ID: 9, Meaning: ShareETA


In [ ]:
import pandas as pd

df_snips = dataset_s["train"].to_pandas()
display(df_snips.head())

,text,label
0,Share my location with Hillary's sister,5
1,Send my current location to my father,5
2,Share my current location with Jim,5
3,Send my location to my husband,5
4,Send my location,5


In [ ]:
df_snips.to_csv("snips_train.csv", index=False)

In [ ]:
SNIPS_MAP = {
    "RequestRide": ("ADD", "TASK"),
    "BookRestaurant": ("ADD", "MEETING"),
    "GetWeather": ("GET", "INFO"),
    "SearchPlace": ("GET", "NOTE"),
    "GetDirections": ("GET", "TASK"),
    "ShareETA": ("UPDATE", "PROGRESS"),
}

In [ ]:
INTENT_MAP = {
    "CREATE_REMINDER": ("ADD",    "TASK"),
    "GET_REMINDER":    ("GET",    "TASK"),
    "DELETE_REMINDER": ("DELETE", "TASK"),
    "CREATE_TIMER":    ("ADD",    "MEETING"),
    "GET_EVENTS":      ("GET",    "MEETING"),
    "SNOOZE_REMINDER": ("UPDATE", "TASK"),
}

SNIPS_MAP = {
    "AddToPlaylist":   ("ADD",    "NOTE"),
    "RateBook":        ("UPDATE", "PROGRESS"),
    "SearchCreativeWork": ("GET", "NOTE"),
}

remind me tmrw morning

uhh can u maybe like play some chill stuff idk

Generated:
"add ML homework deadline Monday"

"add NLP project deadline Tuesday"

"add deep learning report deadline Friday"

Real users:

"yo remind me abt the ml thing monday ish"

"meeting w/ ahmed tmrw 3?"

"that report thing needs to be done by like friday"

"add the thing we talked about"

In [ ]:
from datasets import load_dataset

dataset_a = load_dataset("fathyshalab/atis_intents")

In [ ]:
dataset_a['train'][0]

{'label text': 'atis_flight',
 'text': ' i want to fly from boston at 838 am and arrive in denver at 1110 in the morning',
 'label': 0}

In [ ]:
import pandas as pd

df_atis = dataset_a["train"].to_pandas()

In [ ]:
df_atis.to_csv("atis_intents_train.csv", index=False)

In [ ]:
[
  {
    "id": "S001",
    "text": "add ML homework deadline Monday 11:59 PM",
    "action": "ADD",
    "object": "TASK",
    "tokens": ["add","ML","homework","deadline","Monday","11:59","PM"],
    "ner_tags": ["O","B-TITLE","I-TITLE","O","B-DATE","B-TIME","I-TIME"],
    "has_ner": True,
    "split": "train"
  },
  {
    "id": "S002",
    "text": "show my tasks for today",
    "action": "GET",
    "object": "TASK",
    "tokens": ["show","my","tasks","for","today"],
    "ner_tags": Null,
    "has_ner": False,
    "split": "train"
  },
  {
    "id": "S003",
    "text": "schedule meeting with Ahmed next Tuesday",
    "action": "ADD",
    "object": "MEETING",
    "tokens": ["schedule","meeting","with","Ahmed","next","Tuesday"],
    "ner_tags": ["O","O","O","B-PARTICIPANT","B-DATE","I-DATE"],
    "has_ner": True,
    "split": "val"
  },
  {
    "id": "S004",
    "text": "i finished video 5 in the ML course",
    "action": "UPDATE",
    "object": "PROGRESS",
    "tokens": ["i","finished","video","5","in","the","ML","course"],
    "ner_tags": ["O","O","B-FIELD","B-VALUE","O","O","B-TITLE","I-TITLE"],
    "has_ner": True,
    "split": "train"
  }
]

In [ ]:
NER_TAGS = {
    "O":               0,
    "B-TITLE":         1,
    "I-TITLE":         2,
    "B-DATE":          3,
    "I-DATE":          4,
    "B-TIME":          5,
    "I-TIME":          6,
    "B-PARTICIPANT":   7,
    "I-PARTICIPANT":   8,
    "B-LOCATION":      9,
    "I-LOCATION":      10,
    "B-STATUS":        11,
    "I-STATUS":        12,
    "B-FIELD":         13,
    "I-FIELD":         14,
    "B-VALUE":         15,
    "I-VALUE":         16,
    'B-Des':           17,
    'I-Des':           18
}

Task:
title
date
time
Status

Meeting:
title
Date
Time
PARTICIPANT
LOCATION

Note:
Title
desc

Progress:
title
Field
Value
Date